# Hybrid AI-NWP Multi-Model Forecast Blending System
## Baseline Walkthrough & Research Benchmark

This notebook demonstrates:
1. Loading the configuration and generating deterministic synthetic observations and multi-model forecasts.
2. Executing the baseline Equal-Weight Blend.
3. Computing meteorological verification metrics (MAE, RMSE, Bias, Correlation).
4. Benchmarking individual models vs the blended forecast.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
from configs import get_config
from ingestion.synthetic import SyntheticDataGenerator
from models.baseline import equal_weight_blend
from preprocessing.alignment import align_forecasts_and_observations
from evaluation.comparison import compare_models, compute_skill_rankings

### 1. Generate Synthetic Ground-Truth & Forecasts

In [ ]:
config = get_config()
generator = SyntheticDataGenerator(config)
obs_df, forecasts_df = generator.generate(seed=42)

print(f"Observations records: {len(obs_df):,}")
print(f"Forecast records: {len(forecasts_df):,}")
forecasts_df.head()

### 2. Compute Baseline Equal-Weight Blend

In [ ]:
blended_df = equal_weight_blend(forecasts_df)
combined_forecasts = pd.concat([forecasts_df, blended_df], ignore_index=True)
aligned_df = align_forecasts_and_observations(combined_forecasts, obs_df)
aligned_df.head()

### 3. Evaluate Verification Metrics

In [ ]:
comparison_df = compare_models(aligned_df, group_by=["variable", "lead_time"])
rankings_df = compute_skill_rankings(comparison_df, metric="rmse")
rankings_df[rankings_df["variable"] == "temperature"].head(12)